In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')   # 한국어 문서면 'paraphrase-multilingual-MiniLM-L12-v2'


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
docs = [
    "LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.",
    "임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.",
    "RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.",
    "벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.",
    "파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.",
    "프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.",
]

vecs = model.encode(docs)    # (문서수 × 차원) numpy 배열로 바로 나옴

array([[ 0.0286643 , -0.03401231,  0.01295544, ..., -0.00288203,
        -0.00116074, -0.02507406],
       [ 0.02806919,  0.01722329,  0.00721373, ..., -0.01549055,
        -0.03968976,  0.02745772],
       [-0.00346008,  0.08653531,  0.04054897, ...,  0.00846603,
        -0.03931531,  0.08341915],
       [ 0.01719519,  0.03078108,  0.02184251, ..., -0.02024548,
        -0.02404725,  0.01838731],
       [ 0.03778888,  0.04779398,  0.0465985 , ..., -0.00665872,
        -0.05880586,  0.00478239],
       [ 0.02094554,  0.09014954,  0.02791439, ...,  0.03431499,
        -0.05898093,  0.06769906]], dtype=float32)

In [7]:
query = "가중치를 안 바꾸고 모델 출력을 조절하는 방법"

In [41]:
import numpy as np
q= model.encode(query)
qv = np.linalg.norm(q)
dv = np.linalg.norm(vecs,axis =1 )
cosine = vecs@q/(dv*qv)
idxs = np.argsort(cosine)[::-1][:3]
l = [docs[i ] for i in idxs]
l

['프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.',
 '파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.',
 'RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.']

In [42]:
context = "\n".join(f"- {c}" for c in l)   # 리스트 → 불릿 한 덩어리로 "묶기"

prompt = f"""참고 자료:
{context}

위 자료만 근거로 답하고, 자료에 없으면 "모른다"고 해.
질문: {query}"""

print(prompt)

참고 자료:
- 프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.
- 파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.
- RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.

위 자료만 근거로 답하고, 자료에 없으면 "모른다"고 해.
질문: 가중치를 안 바꾸고 모델 출력을 조절하는 방법
